In [ ]:
!pip install scikit-fuzzy gradio requests folium -q

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import requests
import folium
import gradio as gr
from datetime import datetime
import math
import tempfile, os

In [ ]:
GRAPHHOPPER_API_KEY = "7004e71d-1cf0-4101-9d34-9e5c503911ed"
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"

In [ ]:
def build_fuzzy_system(vehicle: str):
    """
    Xây dựng hệ fuzzy cho từng phương tiện.
    Inputs : distance, weather, time_period, crowd
    Output : fare
    """
    dist_range = {
        'bus':       (0, 20),
        'metro':     (0, 30),
        'taxi':      (0, 40),
        'motorbike': (0, 40),
    }
    fare_range = {
        'bus':       (5,  25),
        'metro':     (10, 40),
        'taxi':      (20, 500),
        'motorbike': (10, 300),
    }

    dist_min, dist_max = dist_range[vehicle]
    fare_min, fare_max = fare_range[vehicle]

    distance    = ctrl.Antecedent(np.linspace(dist_min, dist_max, 200), 'distance')
    weather     = ctrl.Antecedent(np.linspace(0, 3, 100),       'weather')
    time_period = ctrl.Antecedent(np.linspace(0, 2, 100),       'time_period')
    crowd       = ctrl.Antecedent(np.linspace(0, 4, 100),       'crowd')

    fare = ctrl.Consequent(np.linspace(fare_min, fare_max, 200), 'fare')

    # Membership
    # Distance
    span = dist_max - dist_min
    pts  = [dist_min + span * r for r in [0, .1, .2, .35, .45, .55, .65, .8, .9, 1]]
    
    distance['very_short'] = fuzz.trapmf(distance.universe, [pts[0], pts[0], pts[1], pts[3]])
    distance['short']      = fuzz.trimf (distance.universe, [pts[1], pts[3], pts[5]])
    distance['medium']     = fuzz.trimf (distance.universe, [pts[3], pts[5], pts[7]])
    distance['long']       = fuzz.trimf (distance.universe, [pts[5], pts[7], pts[9]])
    distance['very_long']  = fuzz.trapmf(distance.universe, [pts[7], pts[9], pts[9], pts[9]])

    # Weather
    weather['clear']      = fuzz.trimf(weather.universe, [0,   0,   1  ])
    weather['cloudy']     = fuzz.trimf(weather.universe, [0.5, 1,   1.5])
    weather['rain']       = fuzz.trimf(weather.universe, [1,   2,   2.5])
    weather['heavy_rain'] = fuzz.trimf(weather.universe, [2,   3,   3  ])

    # Time 
    time_period['low']    = fuzz.trimf(time_period.universe, [0,   0,   1  ])
    time_period['normal'] = fuzz.trimf(time_period.universe, [0.5, 1,   1.5])
    time_period['peak']   = fuzz.trimf(time_period.universe, [1,   2,   2  ])

    # Crowd
    crowd['very_low']  = fuzz.trimf(crowd.universe, [0, 0, 1])
    crowd['low']       = fuzz.trimf(crowd.universe, [0, 1, 2])
    crowd['medium']    = fuzz.trimf(crowd.universe, [1, 2, 3])
    crowd['high']      = fuzz.trimf(crowd.universe, [2, 3, 4])
    crowd['very_high'] = fuzz.trimf(crowd.universe, [3, 4, 4])

    # Fare 
    f_span = fare_max - fare_min
    fp = [fare_min + f_span * r for r in [0, .15, .3, .5, .7, .85, 1]]

    fare['very_low']  = fuzz.trapmf(fare.universe, [fp[0], fp[0], fp[1], fp[2]])
    fare['low']       = fuzz.trimf (fare.universe, [fp[1], fp[2], fp[3]])
    fare['medium']    = fuzz.trimf (fare.universe, [fp[2], fp[3], fp[4]])
    fare['high']      = fuzz.trimf (fare.universe, [fp[3], fp[4], fp[5]])
    fare['very_high'] = fuzz.trapmf(fare.universe, [fp[4], fp[5], fp[6], fp[6]])

    # Rules 
    rules = [
        # Khoảng cách
        ctrl.Rule(distance['very_short'], fare['very_low']),
        ctrl.Rule(distance['short'],      fare['low']),
        ctrl.Rule(distance['medium'],     fare['medium']),
        ctrl.Rule(distance['long'],       fare['high']),
        ctrl.Rule(distance['very_long'],  fare['very_high']),

        # Cao điểm - Giá tăng
        ctrl.Rule(time_period['peak'] & distance['medium'],    fare['high']),
        ctrl.Rule(time_period['peak'] & distance['long'],      fare['very_high']),
        ctrl.Rule(time_period['peak'] & distance['very_long'], fare['very_high']),

        # Mưa lớn - giá tăng
        ctrl.Rule(weather['heavy_rain'] & distance['medium'],  fare['high']),
        ctrl.Rule(weather['heavy_rain'] & distance['long'],    fare['very_high']),
        ctrl.Rule(weather['rain']       & distance['long'],    fare['high']),

        # Đông người + cao điểm - giá tăng
        ctrl.Rule(crowd['very_high'] & time_period['peak'],    fare['very_high']),
        ctrl.Rule(crowd['high']      & time_period['peak'],    fare['high']),
        ctrl.Rule(crowd['very_low']  & time_period['low'],     fare['very_low']),

        # Trời đẹp + vắng - giá rẻ
        ctrl.Rule(weather['clear'] & crowd['very_low'],        fare['very_low']),
        ctrl.Rule(weather['clear'] & crowd['low'],             fare['low']),
    ]

    system     = ctrl.ControlSystem(rules)
    simulation = ctrl.ControlSystemSimulation(system)
    return simulation, fare


# Build tất cả 4 xe một lần
fuzzy_systems = {v: build_fuzzy_system(v) for v in ['bus', 'metro', 'taxi', 'motorbike']}

In [ ]:
def geocode(address: str):
    """Trả về (lat, lon) từ tên địa điểm. Ưu tiên khu vực TP.HCM."""
    params = {
        'q': address + ', Hồ Chí Minh, Vietnam',
        'format': 'json',
        'limit': 1,
    }
    headers = {'User-Agent': 'FuzzyTransportApp/1.0'}
    try:
        res = requests.get(NOMINATIM_URL, params=params, headers=headers, timeout=10)
        data = res.json()
        if data:
            return float(data[0]['lat']), float(data[0]['lon'])
    except Exception as e:
        print(f"Geocode error: {e}")
    return None, None


def haversine(lat1, lon1, lat2, lon2):
    """Khoảng cách đường chim bay (km)."""
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def get_road_route(lat1, lon1, lat2, lon2, profile='car'):
    """
    Lấy tuyến đường bộ từ GraphHopper Directions API.
    profile: 'car' | 'bike' | 'foot'
    Trả về (distance_km, list_of_[lat,lon]).
    """
    url = 'https://graphhopper.com/api/1/route'
    params = {
        'point': [f'{lat1},{lon1}', f'{lat2},{lon2}'],
        'vehicle': profile,
        'locale': 'vi',
        'key': GRAPHHOPPER_API_KEY,
        'type': 'json',
        'points_encoded': 'false',
    }
    try:
        res = requests.get(url, params=params, timeout=15)
        data = res.json()
        if 'paths' not in data or not data['paths']:
            print(f"GraphHopper: không tìm được đường – {data.get('message','')}")
            return None, None
        path   = data['paths'][0]
        dist_m = path['distance']
        # points.coordinates là list [lon, lat]
        coords = path['points']['coordinates']
        route  = [[c[1], c[0]] for c in coords]  # → [lat, lon]
        return dist_m / 1000, route
    except Exception as e:
        print(f"GraphHopper error: {e}")
        return None, None


def get_time_period(hour=None):
    """Chuyển giờ → giá trị fuzzy [0,2]. Nếu không truyền thì lấy giờ hiện tại."""
    h = hour if hour is not None else datetime.now().hour
    if h in list(range(6,8)) + list(range(16,19)):
        return 2.0, '🔴 Cao điểm (6-8h, 16-19h)'
    elif h in list(range(8,12)) + [12,13]:
        return 1.0, '🟡 Bình thường (8-12h, 12-13h)'
    else:
        return 0.0, '🟢 Thấp điểm (0-6h, 13-16h, 19-24h)'



In [ ]:
# ============================================================
# CELL 6: Core – Tính giá & vẽ bản đồ
# ============================================================

VEHICLE_COLOR = {
    'bus':       '#2563EB',
    'metro':     '#7C3AED',
    'taxi':      '#D97706',
    'motorbike': '#16A34A',
}

VEHICLE_LABEL = {
    'bus':       '🚌 Bus',
    'metro':     '🚇 Metro',
    'taxi':      '🚕 Taxi',
    'motorbike': '🛵 Motorbike',
}


def calculate_fare(vehicle, distance_km, weather_val, time_val, crowd_val):
    sim, fare_var = fuzzy_systems[vehicle]

    # Clamp distance vào đúng universe
    d_max = {'bus': 20, 'metro': 30, 'taxi': 40, 'motorbike': 40}[vehicle]
    sim.input['distance']    = min(distance_km, d_max * 0.99)
    sim.input['weather']     = weather_val
    sim.input['time_period'] = time_val
    sim.input['crowd']       = crowd_val

    try:
        sim.compute()
        raw = sim.output['fare']
        # Nhân lên đơn vị nghìn đồng
        return round(raw * 1000 / 1000) * 1000
    except Exception as e:
        return None


def build_map(vehicle, lat1, lon1, lat2, lon2, route_coords=None):
    center = [(lat1+lat2)/2, (lon1+lon2)/2]
    m = folium.Map(location=center, zoom_start=13,
                   tiles='CartoDB positron')

    color = VEHICLE_COLOR[vehicle]

    # Markers
    folium.Marker([lat1, lon1],
                  popup='📍 Điểm xuất phát',
                  icon=folium.Icon(color='green', icon='home')).add_to(m)
    folium.Marker([lat2, lon2],
                  popup='🏁 Điểm đến',
                  icon=folium.Icon(color='red',   icon='flag')).add_to(m)

    if vehicle == 'metro' or route_coords is None:
        # Đường chim bay
        folium.PolyLine(
            [[lat1, lon1], [lat2, lon2]],
            color=color, weight=4, opacity=0.85,
            dash_array='10 6',
            tooltip='Đường chim bay (Metro)'
        ).add_to(m)
    else:
        # Đường bộ
        folium.PolyLine(
            route_coords,
            color=color, weight=5, opacity=0.9,
            tooltip=f'Tuyến đường – {VEHICLE_LABEL[vehicle]}'
        ).add_to(m)

    # Fit bounds
    m.fit_bounds([[lat1, lon1], [lat2, lon2]])

    tmp = tempfile.NamedTemporaryFile(suffix='.html', delete=False)
    m.save(tmp.name)
    return tmp.name

In [ ]:
WEATHER_MAP = {'☀️ Trời quang': 0, '⛅ Có mây': 1, '🌧️ Mưa nhỏ': 2, '⛈️ Mưa lớn': 3}
CROWD_MAP   = {'Rất thưa': 0, 'Thưa': 1, 'Bình thường': 2, 'Đông': 3, 'Rất đông': 4}
VEHICLE_MAP = {'🚌 Bus': 'bus', '🚇 Metro': 'metro', '🚕 Taxi': 'taxi', '🛵 Motorbike': 'motorbike'}


def run_app(vehicle_label, origin, destination, weather_label, crowd_label, hour):
    vehicle     = VEHICLE_MAP[vehicle_label]
    weather_val = WEATHER_MAP[weather_label]
    crowd_val   = CROWD_MAP[crowd_label]
    time_val, time_label_auto = get_time_period(int(hour))

    # 1. Geocoding
    lat1, lon1 = geocode(origin)
    lat2, lon2 = geocode(destination)

    if lat1 is None or lat2 is None:
        return "❌ Không tìm được địa điểm. Hãy thử nhập rõ hơn (ví dụ: 'Bến Thành, Quận 1').", None

    # 2. Khoảng cách & tuyến đường
    route_coords = None
    if vehicle == 'metro':
        dist_km = haversine(lat1, lon1, lat2, lon2)
        route_type = 'Đường chim bay'
    else:
        profile_map = {'bus': 'car', 'taxi': 'car', 'motorbike': 'bike'}
        dist_km, route_coords = get_road_route(lat1, lon1, lat2, lon2, profile_map[vehicle])
        if dist_km is None:
            dist_km = haversine(lat1, lon1, lat2, lon2) * 1.3  # fallback
            route_type = 'Đường ước tính (fallback)'
        else:
            route_type = 'Đường bộ'

    # 3. Fuzzy fare
    fare = calculate_fare(vehicle, dist_km, weather_val, time_val, crowd_val)

    if fare:
        fare_str = f"{fare:,.0f}đ"
    else:
        fare_str = 'Không tính được'

    result_text = (
        f"## {vehicle_label}  ·  {origin} → {destination}\n\n"
        f"| Thông số | Giá trị |\n"
        f"|---|---|\n"
        f"| 📏 Khoảng cách | **{dist_km:.2f} km** ({route_type}) |\n"
        f"| 🕐 Khung giờ | **{hour:02.0f}:00 – {time_label_auto}** |\n"
        f"| 🌤️ Thời tiết | **{weather_label}** |\n"
        f"| 👥 Mật độ | **{crowd_label}** |\n"
        f"\n### 💰 Giá ước tính: `{fare_str}`"
    )

    # 4. Map
    map_path = build_map(vehicle, lat1, lon1, lat2, lon2, route_coords)

    return result_text, map_path


# Gradio
with gr.Blocks(
    title='Tính Giá Phương Tiện Công Cộng',
    theme=gr.themes.Soft(primary_hue='blue'),
    css="""
        .header-title { text-align:center; font-size:1.8rem; font-weight:700;
                        color:#1e40af; padding:0.5rem 0 0 0; }
        .sub-title     { text-align:center; color:#6b7280; margin-bottom:1rem; }
        footer         { display:none !important; }
    """
) as demo:

    gr.Markdown("<div class='header-title'>🚍 Hệ thống tính giá phương tiện công cộng</div>")
    gr.Markdown("<div class='sub-title'>Sử dụng Fuzzy Logic · Dữ liệu TP. Hồ Chí Minh</div>")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🛞 Chọn phương tiện")
            vehicle_input = gr.Radio(
                choices=list(VEHICLE_MAP.keys()),
                value='🚌 Bus',
                label='Phương tiện',
            )

            gr.Markdown("### 📍 Nhập địa điểm")
            origin_input = gr.Textbox(
                placeholder='Ví dụ: Bến Thành, Quận 1',
                label='Vị trí hiện tại',
            )
            dest_input = gr.Textbox(
                placeholder='Ví dụ: Đại học Bách Khoa, Quận 10',
                label='Địa điểm cần đến',
            )

            gr.Markdown("### ⚙️ Điều kiện")
            weather_input = gr.Dropdown(
                choices=list(WEATHER_MAP.keys()),
                value='☀️ Trời quang',
                label='Thời tiết',
            )
            crowd_input = gr.Dropdown(
                choices=list(CROWD_MAP.keys()),
                value='Bình thường',
                label='Mật độ người',
            )

            now_h = datetime.now().hour
            hour_input = gr.Slider(
                minimum=0, maximum=23, step=1,
                value=now_h,
                label=f'🕐 Chọn giờ khởi hành (hiện tại: {now_h}:00)',
                info='0 = 00:00 · 23 = 23:00'
            )

            confirm_btn = gr.Button('🔍 Xác nhận & Tính giá', variant='primary', size='lg')

        with gr.Column(scale=2):
            result_md  = gr.Markdown(value='*Kết quả sẽ hiển thị ở đây...*')
            map_output = gr.HTML(label='Bản đồ tuyến đường')

    def on_confirm(vehicle, origin, dest, weather, crowd, hour):
        if not origin.strip() or not dest.strip():
            return '⚠️ Vui lòng nhập đầy đủ địa điểm!', ''
        result, map_path = run_app(vehicle, origin, dest, weather, crowd, hour)
        if map_path:
            with open(map_path, 'r', encoding='utf-8') as f:
                map_html = f.read()
            # Embed map trong iframe
            iframe = f'<iframe srcdoc="{map_html.replace(chr(34), chr(39))}" width="100%" height="480px" style="border:none;border-radius:12px;"></iframe>'
            return result, iframe
        return result, ''

    confirm_btn.click(
        fn=on_confirm,
        inputs=[vehicle_input, origin_input, dest_input, weather_input, crowd_input, hour_input],
        outputs=[result_md, map_output],
    )

demo.launch(debug=True, share=True)